# 03 · Vectors, frames and transformations

**Why this is needed:** dot and cross products measure alignment and perpendicularity; frames place the same part in different locations without rebuilding its vertices.

**CAS Computational Form-finding · Block Research Group · ETH Zürich**

A **point** describes a position; a **vector** describes a direction and magnitude.
A **frame** adds an origin and three perpendicular unit axes. We use these objects
to place one box in a local coordinate system without rewriting its vertices.
All coordinates and box dimensions are in metres; unit directions are dimensionless.

**Predict:** if a local x-axis points along world +y, where will local point `(1, 0, 0)` go?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
import numpy as np
from compas.geometry import Box, Frame, Point, Transformation, Vector
from compas_notebook import Viewer
from compas_notebook.config import Config


## 2. Dot and cross products
The dot product measures alignment: perpendicular vectors have dot product zero.
The cross product is perpendicular to both inputs and follows the right-hand rule.
For displacement vectors measured in metres, these products carry m²; the unit
axis vectors used in the next cell are dimensionless.


In [ ]:
u = Vector(2, 0, 0)  # displacement [m]
v = Vector(0, 3, 0)  # displacement [m]
w = u.cross(v)
print(f"u · v = {u.dot(v):g} m²")
print(f"u × v = {list(w)} m²; parallelogram area = {w.length:g} m²")
assert np.isclose(w.dot(u), 0) and np.isclose(w.dot(v), 0)
assert np.isclose(w.length**2 + u.dot(v)**2, u.length**2 * v.length**2)
# Fixed reference example: orthogonal unit axes must form a right-handed basis.
assert list(Vector(1, 0, 0).cross(Vector(0, 1, 0))) == [0, 0, 1]


## 3. Define a local frame
`Frame` normalises its axes and makes them perpendicular. Supply two nonparallel
vectors. Its z-axis follows from x cross y. The frame below moves the origin to
`(2, 1, 0)` and rotates local +x onto world +y.

`Transformation.from_frame_to_frame` actively places geometry from one frame
onto another; its inverse brings it back. This is different from simply relabelling coordinates.


In [ ]:
local_frame = Frame(point=[2, 1, 0], xaxis=[0, 1, 0], yaxis=[-1, 0, 0])
T = Transformation.from_frame_to_frame(Frame.worldXY(), local_frame)
local_point = Point(1, 0, 0)
world_point = local_point.transformed(T)
print("Local point:", list(local_point), "→ world point:", list(world_point))
assert np.allclose(world_point, local_frame.to_world_coordinates(local_point))
assert np.allclose(world_point.transformed(T.inverse()), local_point)
assert np.isclose(local_frame.xaxis.dot(local_frame.yaxis), 0)
assert np.allclose(local_frame.xaxis.cross(local_frame.yaxis), local_frame.zaxis)


## 4. Place the geometry and inspect the scene
The purple box stays at the world origin. The orange copy uses `T`. COMPAS stores
the mesh and frame data; the viewer only displays those objects.


In [ ]:
box = Box(1.2, 0.6, 0.4)
world_mesh = box.to_mesh()
placed_mesh = world_mesh.transformed(T)
assert np.isclose(world_mesh.volume(), placed_mesh.volume())

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [6, -7, 5]
config.view.camera.target = [1, 0.5, 0]
view = Viewer(config=config)
view.scene.add(world_mesh, color="#5552bb", show_edges=True)
view.scene.add(placed_mesh, color="#e89b43", show_edges=True)
view.scene.add(Frame.worldXY())
view.scene.add(local_frame)
view.show()


## 5. Predict → change → check
1. Predict the transformed point before running the frame cell. Starting answer: **(2, 2, 0) m**.
2. Change the frame origin to `(3, 1, 1)`. Which quantities change? Which stay invariant?
3. Reverse the x-axis and supply a compatible y-axis. Use the displayed frame to check your prediction.
4. Explain why the box volume stays unchanged under this transformation.

The next cell makes a separate experiment so **Run All** remains safe.


In [ ]:
experiment_frame = Frame([0, 0, 1], [1, 1, 0], [-1, 1, 0])
experiment_T = Transformation.from_frame_to_frame(Frame.worldXY(), experiment_frame)
p = Point(1, 0, 0).transformed(experiment_T)
assert np.isclose(p.distance_to_point(experiment_frame.point), 1.0)
print("A 1 m local step is still 1 m in world space:", list(p))


## Sources
Original course exercise informed by [ITA21 geometry and frames](https://github.com/compas-teaching/ITA21/tree/7213b322067efe6102e998e83194e8dbe39a82bf) and
[Seoul 2025 COMPAS foundations](https://github.com/BRG-teaching/seoul_2025/tree/b641b732a3e337eb3a19d7a91048946820695e7a). API reference:
[COMPAS geometry](https://compas.dev/compas/2.15.1/api/compas.geometry.html); viewer pattern: [COMPAS JupyterLite](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
